In [3]:
%pip install torch
%pip install numpy
%pip install pandas
%pip install scikit-learn
%pip install anndata


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
# Install dependencies
import pandas as pd
import numpy as np
import subprocess
import os
import anndata as ad

## DeepSEM expects columns to correspond to genes and rows to correspond to cells, so the data frame containing expression values must be transposed before it can be used as input to DeepSEM.

In [6]:
# Read in the expression data
df_expression = pd.read_csv("CHOOSE-multiome-wt-log-norm.csv.gz", compression="gzip")

# Keep track of the original number of rows, columns, and sum of data points to ensure no data is dropped after transposing the data frame
orig_genes = df_expression.shape[0]
orig_cells = df_expression.iloc[:, 1:].shape[1]
orig_sum = np.sum(df_expression.iloc[:, 1:].values)

# Set gene names as the row labels and drop them from the data frame so they are not treated as data
df_expression.index = df_expression.iloc[:, 0]
df_expression = df_expression.drop(columns=[df_expression.columns[0]])

# Prevent pandas from assigning a title to the row labels so it doesn't get added to the CSV file later on
df_expression.index.name = None

# Transpose the data frame
df_transposed = df_expression.T

# Validation: compare the row/col counts
if (orig_genes == df_transposed.shape[1]) and (orig_cells == df_transposed.shape[0]):
    print("Rows/cols preserved.")
else:
    print("Rows/cols do not match.")

# Validation: compare the sums
if orig_sum == np.sum(df_transposed.values):
    print("Numerical data preserved.")
else:
    print("Numerical values are missing.")

Rows/cols preserved.
Numerical data preserved.


## Since we are building a cell type specific GRN, DeepSEM needs the celltype_jf data from the metadata file.

In [7]:
df_meta = pd.read_csv("CHOOSE-multiome-wt-metadata.csv", header=None, skiprows=1)
original_headers = pd.read_csv("CHOOSE-multiome-wt-metadata.csv", nrows=0).columns.tolist()
corrected_headers = ['cell_id'] + original_headers
df_meta.columns = corrected_headers[:df_meta.shape[1]]
df_meta = df_meta.set_index('cell_id')
df_meta.index.name = None
# Proceed with the rest of the alignment to the transposed expression matrix
df_meta_aligned = df_meta.loc[df_transposed.index]
df_cell_types = df_meta_aligned[['celltype_jf']]

## Run DeepSEM with the transposed expression data.

In [13]:
output_dir = 'DeepSEM_Results/'
os.makedirs(output_dir, exist_ok=True)

df_meta_aligned = df_meta_aligned.rename(columns={'celltype_jf': 'cell_type'})

# 2. Create the AnnData object!
# X = the expression matrix, obs = the observations (metadata/cell info)
adata = ad.AnnData(X=df_transposed, obs=df_meta_aligned)

# 3. Save it as an .h5ad file (the native format scanpy and DeepSEM love)
h5ad_file = "CHOOSE_ready_for_DeepSEM.h5ad"
print(f"Saving AnnData object to {h5ad_file}...")
adata.write_h5ad(h5ad_file)

# --- Execute Single DeepSEM Command ---
output_dir = 'DeepSEM_Results/'
os.makedirs(output_dir, exist_ok=True)
save_prefix = os.path.join(output_dir, "DeepSEM_out")

print("Launching DeepSEM celltype_GRN task...")
deepsem_command = [
    "python", "DeepSEM-master/main.py",
    "--task", "celltype_GRN",           # We can finally use the intended task!
    "--data_file", h5ad_file,           # Pass the single .h5ad file
    "--save_name", save_prefix,
    "--setting", "test",                # Keep 'test' to avoid the benchmark crash
    "--n_epochs", "120"
]

try:
    subprocess.run(deepsem_command, check=True)
    print("✅ Successfully generated all cell-type networks!")
except subprocess.CalledProcessError as e:
    print(f"❌ Error running DeepSEM: {e}")

Saving AnnData object to CHOOSE_ready_for_DeepSEM.h5ad...
Launching DeepSEM celltype_GRN task...
dir exist
save dir exist


Traceback (most recent call last):
  File "/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/main.py", line 77, in <module>
    model.train_model()
  File "/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/src/DeepSEM_cell_type_test_specific_GRN_model.py", line 66, in train_model
    vae = VAE_EAD(adj_A_init, 1, self.opt.n_hidden, self.opt.K).float().cuda()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/christina/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1093, in cuda
    return self._apply(lambda t: t.cuda(device))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/christina/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 933, in _apply
    module._apply(fn)
  File "/home/christina/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 933, in _apply
    module._apply(fn)
  File "/home/christina/.venv/lib/python3.12/site-packages/t

❌ Error running DeepSEM: Command '['python', 'DeepSEM-master/main.py', '--task', 'celltype_GRN', '--data_file', 'CHOOSE_ready_for_DeepSEM.h5ad', '--save_name', 'DeepSEM_Results/DeepSEM_out', '--setting', 'test', '--n_epochs', '120']' returned non-zero exit status 1.
